# Beam Search Decoding

**难度:** Medium

实现序列生成的束搜索解码。

束搜索在每一步维护多个候选序列（束），通过扩展和剪枝找到得分最高的序列。

**签名:** `beam_search(log_prob_fn, start_token, max_len, beam_width, eos_token) -> list[int]`

**参数:**
- `log_prob_fn` — 接受 token 序列张量并返回词表上对数概率的可调用对象
- `start_token` — 起始 token 整数
- `beam_width` — 保留的束数量
- `eos_token` — 序列结束 token

**返回:** 最佳序列的 token ID 列表

**约束:**
- 当所有束以 eos 结尾或达到 max_len 时停止
- 返回得分最高的完整序列

**提示:**
1. beams = [(score=0.0, seq=[start_token])]
2. 每步：用所有 token 扩展每个候选 → 按累积分保留前 beam_width 个
3. 所有候选以 eos_token 结尾或达到 max_len 时停止
4. 返回得分最高的序列

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [ ]:
# ✅ INTERVIEW

# ---- 手写 Beam Search ----
# 面试要点：这题不涉及 PyTorch 高级 API，核心是搜索逻辑
# 关键理解：为什么用 log 概率？因为连乘会下溢，累加更稳定

def beam_search(log_prob_fn, start_token, max_len, beam_width, eos_token):
    # 初始化：一个 beam，分数为 0（log 空间），序列只有 start_token
    beams = [(0.0, [start_token])]
    completed = []

    for step in range(max_len):
        candidates = []
        for score, seq in beams:
            # 已完成的序列不再扩展
            if seq[-1] == eos_token:
                completed.append((score, seq))
                continue

            # 获取当前序列下一个 token 的 log 概率分布
            log_probs = log_prob_fn(torch.tensor(seq))

            # 面试考点：topk 的 beam_width 个候选
            topk_lp, topk_idx = log_probs.topk(beam_width)
            for j in range(beam_width):
                # 对数概率累加（等价于概率连乘，但数值更稳定）
                candidates.append((score + topk_lp[j].item(), seq + [topk_idx[j].item()]))

        if not candidates:
            break
        # 剪枝：只保留分数最高的 beam_width 个
        candidates.sort(key=lambda x: x[0], reverse=True)
        beams = candidates[:beam_width]

    # 面试考点：别忘了把未完成的 beam 也加入候选
    all_seqs = completed + beams
    all_seqs.sort(key=lambda x: x[0], reverse=True)
    return all_seqs[0][1]

In [ ]:
# Demo
def simple_fn(tokens):
    lp = torch.full((5,), -10.0)
    lp[min(len(tokens), 4)] = 0.0
    return lp
seq = beam_search(simple_fn, start_token=0, max_len=5, beam_width=2, eos_token=4)
print('Sequence:', seq)

In [ ]:
from torch_judge import check
check('beam_search')

## 📝 核心思路总结

1. **束搜索 = 宽度优先的受限搜索**：每步保留 beam_width 个最优候选，平衡搜索空间与效率
2. **用 log 概率而非概率**：避免连乘导致的数值下溢，log 概率累加即可
3. **已完成的序列单独收集**：遇到 eos_token 的序列移入 completed 列表，不再扩展